# COS Method Improved
This notebook revisits Fang & Oosterlee (2008) through the lens of Junike (2024) and Junike-Pankrashkin (2022). The point is not a new pricer; it is to make COS choose the truncation range and the number of cosine terms as a coupled numerical policy.

The full-paper replication shows why that matters. Under the naive FO2008-style setup, BSM Table 2 develops a flat observed error floor, Heston Table 5 converges too slowly at long maturity, and the Heston strip becomes sensitive to the shared interval and the strip-reference construction. Those are exactly the regimes where support truncation, series truncation, and coefficient-side instability need to be separated rather than lumped together as one vague “COS error”.

In the baseline FO2008 setup, the interval is usually built from low-order cumulants using

$$
[a,b] = \left[c_1 - L\sqrt{c_2 + \sqrt{|c_4|}},\; c_1 + L\sqrt{c_2 + \sqrt{|c_4|}}\right].
$$

That rule is convenient, but it is still just a rule of thumb. If $[a,b]$ is too short, the method throws away tail mass before the cosine expansion even starts. After that, raising $N$ only gives a better approximation on the truncated support; it does not recover the missing mass.

The Junike-Pankrashkin fix is to choose the interval from a tail tolerance instead of from a fixed cumulant multiplier. Pick a center $m$ and a half-width $M$, and require the tail outside $[m-M,m+M]$ to be small. By Markov's inequality,

$$
\mathbb{P}(|X-m| \ge M) \le \frac{\mathbb{E}[|X-m|^n]}{M^n},
$$

so it is enough to choose

$$
M \ge \left(\frac{\mathbb{E}[|X-m|^n]}{\varepsilon}\right)^{1/n}, \qquad [a,b] = [m-M,\; m+M].
$$

Junike (2024) then handles the second knob: once the support is fixed, choose enough cosine modes to resolve it. That is the logic behind `COSGridPolicy`, centered intervals, and the adaptive `cos_improved` path used below.

In [1]:
# Cell 1 — Imports and paths
from __future__ import annotations

import pathlib
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

CWD = pathlib.Path.cwd().resolve()
for _candidate in [CWD, CWD.parent, CWD.parent.parent]:
    if (_candidate / 'src' / 'foureng').exists() and (_candidate / 'benchmarks').exists():
        REPO_ROOT = _candidate
        break
else:
    raise RuntimeError(f'Could not locate repo root from {CWD}')

for _path in (REPO_ROOT, REPO_ROOT / 'src'):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

OUTDIR = REPO_ROOT / 'benchmarks' / 'cos_method_improved' / 'outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)

from benchmarks.paper_replications.fo2008_cos.params import CASES, PaperCase
from foureng.models.base import ForwardSpec
from foureng.models.bsm import BsmParams, bsm_cf, bsm_cumulants
from foureng.models.cgmy import CgmyParams, cgmy_cf, cgmy_cumulants
from foureng.models.heston import HestonParams, heston_cf, heston_cumulants
from foureng.models.variance_gamma import VGParams, vg_cf, vg_cumulants
from foureng.pipeline import price_strip
from foureng.pricers.cos import cos_adaptive_decision, cos_auto_grid, cos_prices, recommended_cos_policy
from foureng.pricers.lewis import lewis_call_prices
from foureng.utils.cumulants import Cumulants, cos_centered_half_width
from foureng.utils.grids import COSGrid, COSGridPolicy

pd.set_option('display.float_format', lambda x: f'{x: .6e}')
warnings.filterwarnings('ignore', category=RuntimeWarning)
print('repo root:', REPO_ROOT)
print('output dir:', OUTDIR)


repo root: /Users/nigelli/Desktop/Columbia MAFN/26Spring/MATH5030/Project/5030_Carr-Madan
output dir: /Users/nigelli/Desktop/Columbia MAFN/26Spring/MATH5030/Project/5030_Carr-Madan/benchmarks/cos_method_improved/outputs


In [2]:
# Cell 2 — Helpers
N_REP = 5


def timed_median_ms(fn, *args, n_rep=N_REP, **kwargs):
    fn(*args, **kwargs)
    times = []
    out = None
    for _ in range(n_rep):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1e3)
    return out, float(np.median(times))


def fwd_for(case: PaperCase) -> ForwardSpec:
    r = case.params.get('r', 0.0)
    q = case.params.get('q', 0.0)
    return ForwardSpec(S0=case.forward, r=r, q=q, T=case.maturity)


def params_for(case: PaperCase):
    p = {k: v for k, v in case.params.items() if k not in ('r', 'q')}
    if case.model == 'bsm':
        return BsmParams(**p)
    if case.model == 'heston':
        return HestonParams(**p)
    if case.model == 'vg':
        return VGParams(**p)
    if case.model == 'cgmy':
        return CgmyParams(**p)
    raise ValueError(case.model)


def cf_for(case: PaperCase):
    fwd = fwd_for(case)
    p = params_for(case)
    if case.model == 'bsm':
        return lambda u: bsm_cf(u, fwd, p), fwd, p
    if case.model == 'heston':
        return lambda u: heston_cf(u, fwd, p), fwd, p
    if case.model == 'vg':
        return lambda u: vg_cf(u, fwd, p), fwd, p
    if case.model == 'cgmy':
        return lambda u: cgmy_cf(u, fwd, p), fwd, p
    raise ValueError(case.model)


def cumulants_for(case: PaperCase):
    fwd = fwd_for(case)
    p = params_for(case)
    if case.model == 'bsm':
        return bsm_cumulants(fwd, p)
    if case.model == 'heston':
        return heston_cumulants(fwd, p)
    if case.model == 'vg':
        return vg_cumulants(fwd, p)
    if case.model == 'cgmy':
        return cgmy_cumulants(fwd, p)
    raise ValueError(case.model)


def bs_call_exact(strikes, fwd: ForwardSpec, sigma: float):
    strikes = np.asarray(strikes, dtype=float)
    vol = sigma * np.sqrt(fwd.T)
    d1 = (np.log(fwd.F0 / strikes) + 0.5 * sigma * sigma * fwd.T) / vol
    d2 = d1 - vol
    return fwd.disc * (fwd.F0 * norm.cdf(d1) - strikes * norm.cdf(d2))


def reference_for_case(case: PaperCase):
    phi, fwd, p = cf_for(case)
    K = np.asarray(case.strikes, dtype=float)
    if case.model == 'bsm':
        return np.asarray(bs_call_exact(K, fwd, p.sigma), dtype=float)
    if case.model == 'heston':
        return np.asarray(
            lewis_call_prices(
                phi,
                K,
                spot=fwd.S0,
                texp=fwd.T,
                intr=fwd.r,
                divr=fwd.q,
                method='trapz',
                u_max=250.0,
                n_u=8192,
            ),
            dtype=float,
        )
    ref = np.asarray(case.reference_values, dtype=float)
    return ref.reshape(-1)


def max_abs_err(prices, ref):
    return float(np.max(np.abs(np.asarray(prices, dtype=float) - np.asarray(ref, dtype=float))))


def paper_grid_for(case: PaperCase):
    cums = cumulants_for(case)
    if case.model == 'cgmy' and 'trunc_ab' in case.extras:
        a, b = case.extras['trunc_ab']
        return COSGrid(N=max(case.Ns), a=float(a), b=float(b), label='paper')
    L = float(case.extras.get('L', 10.0))
    return cos_auto_grid(cums, N=max(case.Ns), L=L)


In [3]:
# Cell 3 — Fixed COS vs paper-grid COS vs improved adaptive COS
CASE_IDS = [
    'bsm_table2',
    'heston_table4_t1',
    'heston_table5_t10',
    'heston_table6_strip',
    'vg_table7_t01',
    'vg_table7_t1',
    'cgmy_table8_y05',
    'cgmy_table10_y198',
]


def cmp_status(lhs, rhs, *, rtol=1e-12, atol=1e-18):
    if pd.isna(lhs) or pd.isna(rhs):
        return 'n/a'
    if np.isclose(lhs, rhs, rtol=rtol, atol=atol):
        return 'match'
    return 'better' if lhs < rhs else 'worse'


rows = []
compare_rows = []
for cid in CASE_IDS:
    case = CASES[cid]
    phi, fwd, p = cf_for(case)
    K = np.asarray(case.strikes, dtype=float)
    ref = reference_for_case(case)
    cums = cumulants_for(case)

    def _default_cos():
        return price_strip(case.model, 'cos', K, fwd, p)

    def _paper_cos():
        return cos_prices(phi, fwd, K, paper_grid_for(case)).call_prices

    policy = recommended_cos_policy(case.model, p, mode='benchmark')
    decision = cos_adaptive_decision(cums, model=case.model, params=p, policy=policy, strike_count=len(K))

    def _improved_cos():
        return price_strip(case.model, 'cos_improved', K, fwd, p, grid=policy)

    default_prices, default_ms = timed_median_ms(_default_cos)
    paper_prices, paper_ms = timed_median_ms(_paper_cos)
    improved_prices, improved_ms = timed_median_ms(_improved_cos)

    default_err = max_abs_err(default_prices, ref)
    paper_grid_err = max_abs_err(paper_prices, ref)
    improved_err = max_abs_err(improved_prices, ref)

    rows.append({
        'case_id': cid,
        'model': case.model,
        'n_strikes': len(K),
        'default_err': default_err,
        'paper_grid_err': paper_grid_err,
        'improved_err': improved_err,
        'default_ms': default_ms,
        'paper_grid_ms': paper_ms,
        'improved_ms': improved_ms,
        'improved_method': decision.method,
        'improved_N': decision.grid.N,
        'improved_width': decision.grid.width,
        'improved_dx': decision.grid.dx,
        'improved_center': decision.grid.center,
        'improved_L': decision.L_used,
        'tail_proxy': decision.tail_proxy,
        'tail_family': decision.tail_family,
        'decision_reason': decision.reason,
    })

    paper_best_idx = int(np.argmin(case.paper_errors)) if case.paper_errors is not None else None
    paper_best_n = case.Ns[paper_best_idx] if paper_best_idx is not None else np.nan
    paper_best_err = case.paper_errors[paper_best_idx] if paper_best_idx is not None else np.nan
    paper_best_ms = (
        case.paper_times_ms[paper_best_idx]
        if paper_best_idx is not None and case.paper_times_ms is not None
        else np.nan
    )

    compare_rows.append({
        'case_id': cid,
        'paper_best_N': paper_best_n,
        'paper_best_err': paper_best_err,
        'paper_best_ms_historical': paper_best_ms,
        'default_err': default_err,
        'paper_grid_err': paper_grid_err,
        'improved_method': decision.method,
        'improved_N': decision.grid.N,
        'improved_err': improved_err,
        'improved_ms': improved_ms,
        'vs_default': cmp_status(improved_err, default_err),
        'vs_paper_grid': cmp_status(improved_err, paper_grid_err),
        'vs_paper_best': cmp_status(improved_err, paper_best_err),
    })

SUMMARY_DF = pd.DataFrame(rows)
SUMMARY_DF['gain_vs_default'] = SUMMARY_DF['default_err'] / SUMMARY_DF['improved_err']
SUMMARY_DF['gain_vs_paper_grid'] = SUMMARY_DF['paper_grid_err'] / SUMMARY_DF['improved_err']
SUMMARY_DF.to_csv(OUTDIR / 'cos_method_improved_summary.csv', index=False)

PAPER_COMPARE_DF = pd.DataFrame(compare_rows)
PAPER_COMPARE_DF.to_csv(OUTDIR / 'cos_method_improved_paper_compare.csv', index=False)

display(SUMMARY_DF.sort_values('improved_err'))
display(PAPER_COMPARE_DF)
print('wrote', OUTDIR / 'cos_method_improved_summary.csv')
print('wrote', OUTDIR / 'cos_method_improved_paper_compare.csv')


,case_id,model,n_strikes,default_err,paper_grid_err,improved_err,default_ms,paper_grid_ms,improved_ms,improved_method,improved_N,improved_width,improved_dx,improved_center,improved_L,tail_proxy,tail_family,decision_reason,gain_vs_default,gain_vs_paper_grid
0,bsm_table2,bsm,3,1.598721e-14,1.598721e-14,1.541128e-14,7.220800e-02,9.045799e-02,1.285000e-01,cos,64,1.264911e+00,1.976424e-02,-3.125000e-03,8.000000e+00,1.266417e-14,gaussian_like,,1.037371e+00,1.037371e+00
1,heston_table4_t1,heston,1,6.099287e-08,6.566175e-07,2.217959e-11,1.002080e-01,8.841700e-02,1.282500e-01,cos,512,5.498447e+00,1.073915e-02,-1.428989e-02,8.000000e+00,1.033446e-52,gaussian_like,,2.749954e+03,2.960458e+04
7,cgmy_table10_y198,cgmy,1,2.464162e-11,1.465139e-11,6.407674e-11,7.191600e-02,4.883300e-02,7.160410e-01,lewis,8192,2.743881e+02,3.349464e-02,-4.787935e+01,1.400000e+01,1.964589e-05,heavy,interval width 274.39 exceeded 40.00; switched...,3.845642e-01,2.286538e-01
2,heston_table5_t10,heston,1,5.073275e-12,4.684388e-03,9.678303e-11,9.375000e-02,8.800000e-02,2.095830e-01,cos,1024,1.772249e+01,1.730712e-02,-1.919287e-01,8.000000e+00,5.328217e-37,gaussian_like,,5.241906e-02,4.840092e+07
6,cgmy_table8_y05,cgmy,1,1.187175e-10,2.155218e-10,1.187423e-10,6.958299e-02,5.775000e-02,1.518340e-01,cos,1024,2.730626e+01,2.666627e-02,-8.027873e-02,2.441406e+01,6.775002e-12,semi_heavy,,9.997906e-01,1.815038e+00
5,vg_table7_t1,vg,1,1.990692e-10,4.389591e-10,1.996909e-10,5.179201e-02,4.754201e-02,1.759170e-01,cos,2048,3.741831e+00,1.827066e-03,-8.932966e-03,1.000000e+01,2.071553e-05,heavy,,9.968866e-01,2.198192e+00
3,heston_table6_strip,heston,21,6.977499e-08,2.622245e-06,2.922036e-10,2.961250e-01,2.162920e-01,3.315840e-01,cos,512,5.498447e+00,1.073915e-02,-1.428989e-02,8.000000e+00,1.033446e-52,gaussian_like,,2.387890e+02,8.974036e+03
4,vg_table7_t01,vg,1,4.444359e-05,4.943326e-08,1.490185e-08,6.233300e-02,1.425840e-01,1.073750e-01,cos,1024,1.686144e+00,1.646625e-03,-8.932966e-04,1.000000e+01,5.449579e-06,heavy,,2.982422e+03,3.317257e+00


,case_id,paper_best_N,paper_best_err,paper_best_ms_historical,default_err,paper_grid_err,improved_method,improved_N,improved_err,improved_ms,vs_default,vs_paper_grid,vs_paper_best
0,bsm_table2,64,3.550000e-15,3.270000e-02,1.598721e-14,1.598721e-14,cos,64,1.541128e-14,1.285000e-01,better,better,worse
1,heston_table4_t1,200,3.700000e-09,1.539000e-01,6.099287e-08,6.566175e-07,cos,512,2.217959e-11,1.282500e-01,better,better,better
2,heston_table5_t10,140,9.880000e-10,1.230000e-01,5.073275e-12,4.684388e-03,cos,1024,9.678303e-11,2.095830e-01,worse,better,better
3,heston_table6_strip,200,2.050000e-08,4.214000e-01,6.977499e-08,2.622245e-06,cos,512,2.922036e-10,3.315840e-01,better,better,better
4,vg_table7_t01,2048,7.980000e-08,NaN,4.444359e-05,4.943326e-08,cos,1024,1.490185e-08,1.073750e-01,better,better,better
5,vg_table7_t1,150,1.510000e-09,NaN,1.990692e-10,4.389591e-10,cos,2048,1.996909e-10,1.759170e-01,worse,better,better
6,cgmy_table8_y05,140,4.040000e-09,1.216000e-01,1.187175e-10,2.155218e-10,cos,1024,1.187423e-10,1.518340e-01,worse,better,better
7,cgmy_table10_y198,40,1.940000e-15,5.380000e-02,2.464162e-11,1.465139e-11,lewis,8192,6.407674e-11,7.160410e-01,worse,worse,worse


wrote /Users/nigelli/Desktop/Columbia MAFN/26Spring/MATH5030/Project/5030_Carr-Madan/benchmarks/cos_method_improved/outputs/cos_method_improved_summary.csv
wrote /Users/nigelli/Desktop/Columbia MAFN/26Spring/MATH5030/Project/5030_Carr-Madan/benchmarks/cos_method_improved/outputs/cos_method_improved_paper_compare.csv


In [4]:
# Cell 4 — Heston T=10 diagnostics: support error, series error, coefficient error
case = CASES['heston_table5_t10']
phi, fwd, p = cf_for(case)
K = np.asarray(case.strikes, dtype=float)
ref = float(reference_for_case(case)[0])
cums = cumulants_for(case)
c1, c2, c4 = cums
cobj = Cumulants(c1=c1, c2=c2, c4=c4)

series_rows = []
for N in [64, 128, 256, 512, 1024, 2048, 4096]:
    half_width = cos_centered_half_width(cobj, L=32.0)
    grid = COSGrid(N=N, a=-half_width, b=half_width, center=c1, label='paper_centered')
    p_put = float(cos_prices(phi, fwd, K, grid, payoff_mode='put_parity').call_prices[0])
    p_call = float(cos_prices(phi, fwd, K, grid, payoff_mode='call_direct').call_prices[0])
    series_rows.append({
        'N': N,
        'width': grid.width,
        'dx': grid.dx,
        'put_parity_err': abs(p_put - ref),
        'call_direct_err': abs(p_call - ref),
        'direct_minus_put': abs(p_call - p_put),
    })
SERIES_DF = pd.DataFrame(series_rows)
SERIES_DF.to_csv(OUTDIR / 'heston_t10_series_vs_coeff_error.csv', index=False)

support_rows = []
for L in [6, 8, 10, 12, 16, 24, 32]:
    pol = COSGridPolicy(
        mode='benchmark',
        truncation='heuristic',
        centered=True,
        L=float(L),
        fixed_N=4096,
        width_fallback=0.0,
    )
    dec = cos_adaptive_decision(cums, model=case.model, params=p, policy=pol)
    price = float(cos_prices(phi, fwd, K, dec.grid, payoff_mode='put_parity').call_prices[0])
    support_rows.append({
        'L': float(L),
        'width': dec.grid.width,
        'tail_proxy': dec.tail_proxy,
        'err': abs(price - ref),
    })
SUPPORT_DF = pd.DataFrame(support_rows)
SUPPORT_DF.to_csv(OUTDIR / 'heston_t10_support_error.csv', index=False)

policy_rows = []
for name, pol in [
    ('heuristic', COSGridPolicy(mode='benchmark', truncation='heuristic', centered=True, L=10.0, width_fallback=0.0)),
    ('tolerance', COSGridPolicy(mode='benchmark', truncation='tolerance', centered=True, eps_trunc=1e-10, width_fallback=0.0)),
    ('paper', COSGridPolicy(mode='benchmark', truncation='paper', centered=True, paper_L=32.0, width_fallback=0.0)),
]:
    dec = cos_adaptive_decision(cums, model=case.model, params=p, policy=pol)
    price = float(cos_prices(phi, fwd, K, dec.grid, payoff_mode='put_parity').call_prices[0])
    policy_rows.append({
        'policy': name,
        'N': dec.grid.N,
        'width': dec.grid.width,
        'dx': dec.grid.dx,
        'L_used': dec.L_used,
        'tail_proxy': dec.tail_proxy,
        'err': abs(price - ref),
    })
POLICY_DF = pd.DataFrame(policy_rows)
POLICY_DF.to_csv(OUTDIR / 'heston_t10_policy_comparison.csv', index=False)

display(POLICY_DF)
display(SUPPORT_DF)
display(SERIES_DF)


,policy,N,width,dx,L_used,tail_proxy,err
0,heuristic,2048,2.215311e+01,1.081695e-02,1.000000e+01,2.102734e-57,5.062617e-12
1,tolerance,1024,1.772249e+01,1.730712e-02,8.000000e+00,5.328217e-37,1.842729e-09
2,paper,4096,7.088995e+01,1.730712e-02,3.200000e+01,0.000000e+00,2.842171e-14


,L,width,tail_proxy,err
0,6.000000e+00,1.329186e+01,3.946411e-21,6.998943e-07
1,8.000000e+00,1.772249e+01,5.328217e-37,1.842729e-09
2,1.000000e+01,2.215311e+01,2.102734e-57,5.062617e-12
3,1.200000e+01,2.658373e+01,2.425545e-82,7.105427e-15
4,1.600000e+01,3.544497e+01,8.059865e-146,4.263256e-14
5,2.400000e+01,5.316746e+01,0.000000e+00,4.263256e-14
6,3.200000e+01,7.088995e+01,0.000000e+00,2.842171e-14


,N,width,dx,put_parity_err,call_direct_err,direct_minus_put
0,64,7.088995e+01,1.107655e+00,8.128869e-01,1.334703e+14,1.334703e+14
1,128,7.088995e+01,5.538277e-01,1.493128e-04,8.041030e+10,8.041030e+10
2,256,7.088995e+01,2.769138e-01,1.330178e-05,9.932138e+09,9.932138e+09
3,512,7.088995e+01,1.384569e-01,6.711893e-10,9.793140e+04,9.793140e+04
4,1024,7.088995e+01,6.922846e-02,2.842171e-14,5.357175e+01,5.357175e+01
5,2048,7.088995e+01,3.461423e-02,2.842171e-14,5.357172e+01,5.357172e+01
6,4096,7.088995e+01,1.730712e-02,2.842171e-14,5.357172e+01,5.357172e+01


In [5]:
# Cell 5 — Plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(SUPPORT_DF['L'], SUPPORT_DF['err'], marker='o', label='support error')
axes[0].set_yscale('log')
axes[0].set_xlabel('L')
axes[0].set_ylabel('abs error')
axes[0].set_title('Heston T=10: support truncation')
axes[0].grid(True, alpha=0.3)

axes[1].plot(SERIES_DF['N'], SERIES_DF['put_parity_err'], marker='o', label='put + parity')
axes[1].plot(SERIES_DF['N'], SERIES_DF['call_direct_err'], marker='o', label='call direct')
axes[1].set_xscale('log', base=2)
axes[1].set_yscale('log')
axes[1].set_xlabel('N')
axes[1].set_ylabel('abs error')
axes[1].set_title('Heston T=10: coefficient-side error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

x = np.maximum(SUMMARY_DF['default_err'].to_numpy(), 1e-18)
y = np.maximum(SUMMARY_DF['improved_err'].to_numpy(), 1e-18)
axes[2].scatter(x, y)
for _, row in SUMMARY_DF.iterrows():
    axes[2].annotate(row['case_id'], (max(row['default_err'], 1e-18), max(row['improved_err'], 1e-18)), fontsize=8)
axes[2].plot([x.min(), x.max()], [x.min(), x.max()], linestyle='--', color='black', alpha=0.5)
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_xlabel('default abs error')
axes[2].set_ylabel('improved abs error')
axes[2].set_title('Case summary')
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(OUTDIR / 'cos_method_improved_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()
print('wrote', OUTDIR / 'cos_method_improved_diagnostics.png')


wrote /Users/nigelli/Desktop/Columbia MAFN/26Spring/MATH5030/Project/5030_Carr-Madan/benchmarks/cos_method_improved/outputs/cos_method_improved_diagnostics.png


/var/folders/tl/hb3sh0p16wb8f6cdsnkjq_wm0000gn/T/ipykernel_3893/3211781427.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Cell 6 — Concise takeaways
best = SUMMARY_DF[['case_id', 'improved_method', 'improved_N', 'improved_width', 'improved_err', 'gain_vs_default']].copy()
print('Improved COS summary')
print(best.to_string(index=False))
print('\nPaper comparison')
print(PAPER_COMPARE_DF.to_string(index=False))
print('\nCounts')
print('beat default:', int((PAPER_COMPARE_DF['vs_default'] == 'better').sum()), 'of', len(PAPER_COMPARE_DF))
print('beat paper-grid replay:', int((PAPER_COMPARE_DF['vs_paper_grid'] == 'better').sum()), 'of', len(PAPER_COMPARE_DF))
print('beat paper best reported error:', int((PAPER_COMPARE_DF['vs_paper_best'] == 'better').sum()), 'of', len(PAPER_COMPARE_DF))
print('\nNotes')
print('- The adaptive path is not uniformly better than the old default on every single case; it is mainly a robustness fix, not a guarantee that one frozen benchmark will always win.')
print('- It does beat the paper-best reported error on most cases in this notebook, but BSM remains reference-limited and CGMY Y=1.98 is intentionally treated as a hostile fallback regime.')
print('- Paper milliseconds stay in the table as historical context only; they are not portable timing claims.')


Improved COS summary
            case_id improved_method  improved_N  improved_width  improved_err  gain_vs_default
         bsm_table2             cos          64    1.264911e+00  1.541128e-14     1.037371e+00
   heston_table4_t1             cos         512    5.498447e+00  2.217959e-11     2.749954e+03
  heston_table5_t10             cos        1024    1.772249e+01  9.678303e-11     5.241906e-02
heston_table6_strip             cos         512    5.498447e+00  2.922036e-10     2.387890e+02
      vg_table7_t01             cos        1024    1.686144e+00  1.490185e-08     2.982422e+03
       vg_table7_t1             cos        2048    3.741831e+00  1.996909e-10     9.968866e-01
    cgmy_table8_y05             cos        1024    2.730626e+01  1.187423e-10     9.997906e-01
  cgmy_table10_y198           lewis        8192    2.743881e+02  6.407674e-11     3.845642e-01

Paper comparison
            case_id  paper_best_N  paper_best_err  paper_best_ms_historical   default_err  paper_grid_err 